<a href="https://colab.research.google.com/github/columbia-data-club/meetings/blob/main/2025/april_02_data_engineering_with_polars_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![A blue background with the pandas logo and the words Columbia Data Club on it](https://raw.githubusercontent.com/columbia-data-club/meetings/main/assets/images/2025/polars.png)

# Python Data Engineering with Polars II

April 2, 2025

by [Moacir P. de Sá Pereira](https://moacir.com) for the [Columbia Data Club](https://github.com/columbia-data-club/)


This notebook builds on [an introduction to data engineering](https://github.com/columbia-data-club/meetings/blob/main/2025/march_5_data_engineering_with_polars_1.ipynb) with [Polars](https://pola.rs). A basic understanding of Python syntax (such as the one covered in the Data Club’s [Intro to Python video](https://youtu.be/l45rzo4MUHs)) should suffice.

We will continue looking to our perennial favorite today, [NYC Yellow Cab trip data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

## Fire up the Data

Last time, we broadly discussed the two Polars mental models for data engineering: **contexts** and **expressions**. As a reminder:

* An **expression** is a lazy representation of a data transformation. We will be focusing on those today.
* A **context** is, well, a context in which an expression is evaluated. We mostly concern ourselves with four of them:
  * `select` chooses a subset of columns to transform and/or return
  * `with_columns` appends columns to already existing columns
  * `filter` filters rows based on certain criteria
  * `group_by` groups rows based on certain criteria, typically for aggregating.

Importantly, because expressions are lazy, we can assign them to variables. But let’s load the data and have a look at the columns again.

In [3]:
import polars as pl

df = pl.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet")
df.glimpse()

Rows: 3668371
Columns: 19
$ VendorID                       <i32> 2, 2, 2, 2, 2, 1, 2, 1, 1, 2
$ tpep_pickup_datetime  <datetime[μs]> 2024-12-01 00:12:27, 2024-11-30 23:56:04, 2024-12-01 00:50:35, 2024-12-01 00:18:16, 2024-12-01 00:56:13, 2024-12-01 00:21:17, 2024-12-01 00:04:53, 2024-12-01 00:15:28, 2024-12-01 00:38:54, 2024-12-01 00:00:21
$ tpep_dropoff_datetime <datetime[μs]> 2024-12-01 00:31:12, 2024-12-01 00:28:15, 2024-12-01 01:24:46, 2024-12-01 00:33:16, 2024-12-01 01:18:25, 2024-12-01 00:37:22, 2024-12-01 00:31:03, 2024-12-01 00:20:13, 2024-12-01 01:03:46, 2024-12-01 00:05:27
$ passenger_count                <i64> 1, 1, 4, 3, 1, 1, 1, 1, 1, 2
$ trip_distance                  <f64> 9.76, 7.62, 20.07, 2.34, 5.05, 4.3, 7.66, 0.3, 9.4, 0.72
$ RatecodeID                     <i64> 1, 1, 2, 1, 1, 1, 1, 1, 1, 1
$ store_and_fwd_flag             <str> 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N'
$ PULocationID                   <i32> 138, 158, 132, 142, 107, 249, 186, 148, 234, 211
$ 

Last time we created an expression for deriving the tip percentage. Let’s do that again just to refresh what expressions look like.

In [11]:
tip_pct = (pl.col("tip_amount") / pl.col("fare_amount")).round(2)
df.select(
    "tip_amount",
    "fare_amount",
    tip_pct.alias("tip_pct")
).glimpse()

Rows: 3668371
Columns: 3
$ tip_amount  <f64> 4.72, 8.46, 0.0, 4.12, 5.0, 5.1, 8.04, 0.0, 0.0, 2.44
$ fare_amount <f64> 38.0, 37.3, 70.0, 15.6, 26.8, 20.5, 35.2, 5.8, 39.4, 7.2
$ tip_pct     <f64> 0.12, 0.23, 0.0, 0.26, 0.19, 0.25, 0.23, 0.0, 0.0, 0.34



## We Are Low-Key EDAing Here?

Absolutely. We are using these contexts to help us understand our data better, with an eye toward wrangling it for future analysis.

What are some assumptions about our data we should test and perhaps correct?

In [ ]:
import polars as pl
from datetime import datetime as dt